In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
# from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import joblib
from xgboost import XGBRegressor

Sequence to predict and know about ETA prediction model
1. Load data
2. Inspect target
3. Check available features
4. Build ETA features
5. Remove leakage
6. Train/test split
7. Preprocess categorical features
8. Train XGBoost Regressor
9. Evaluate MAE/RMSE/R²
10. SHAP / feature importance
11. Save model + preprocessor
12. Build eta_predictor.py
13. Integrate into simulation

In [2]:
df = pd.read_csv("../datasets/processed/dataco_features.csv")

print("Shape:", df.shape)
df.head()

Shape: (180519, 50)


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,product_name,product_price,shipping_date_dateorders,shipping_mode,order_hour,order_day_of_week,order_month,is_weekend,shipping_delay_days,shipping_mode_delay_rate
0,CASH,2,4,88.790001,239.979996,Advance shipping,0,43,Camping & Hiking,Hickory,...,Diamondback Women's Serene Classic Comfort Bi,299.980011,2015-01-03 00:00:00,Standard Class,0,3,1,0,-2,NaN
1,PAYMENT,3,4,91.180000,193.990005,Advance shipping,0,48,Water Sports,Chicago,...,Pelican Sunstream 100 Kayak,199.990005,2015-01-04 00:21:00,Standard Class,0,3,1,0,-1,0.0
2,PAYMENT,3,4,68.250000,227.500000,Advance shipping,0,24,Women's Apparel,Chicago,...,Nike Men's Dri-FIT Victory Golf Polo,50.000000,2015-01-04 00:21:00,Standard Class,0,3,1,0,-1,0.0
3,PAYMENT,3,4,36.470001,107.889999,Advance shipping,0,18,Men's Footwear,Chicago,...,Nike Men's CJ Elite 2 TD Football Cleat,129.990005,2015-01-04 00:21:00,Standard Class,0,3,1,0,-1,0.0
4,CASH,5,4,33.590000,159.940002,Late delivery,1,46,Indoor/Outdoor Games,San Antonio,...,O'Brien Men's Neoprene Life Vest,49.980000,2015-01-06 01:03:00,Standard Class,1,3,1,0,1,0.0


In [3]:
print(df.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_dateorders', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_name', 'product_price', 'shipping_date_dateorders', 'shipping_mode', 'order_hour', 'order_day_of_week', 'order_month', 'is_weekend', 'shipping_delay_days', 'shipping_mode_delay_rate']


In [12]:
print("Target:", "days_for_shipping_real")

print("\nTarget statistics:")
print(df["days_for_shipping_real"].describe())

print("\nMissing values:")
print(df["days_for_shipping_real"].isna().sum())

print("\nUnique target values:")
print(sorted(df["days_for_shipping_real"].unique()))

Target: days_for_shipping_real

Target statistics:
count    180519.000000
mean          3.497654
std           1.623722
min           0.000000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: days_for_shipping_real, dtype: float64

Missing values:
0

Unique target values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


In [33]:
ETA_FEATURES = [
    "shipping_mode",
    "days_for_shipment_scheduled",
    "order_item_quantity",
    "product_price",
    "customer_segment",
    "market",
    "latitude",
    "longitude",
    "shipping_mode_delay_rate",
    "order_hour",
    "order_day_of_week",
    "order_month",
    "is_weekend",
    "order_region",
    "sales",
    "order_profit_per_order",
    "order_item_profit_ratio"
]

df[ETA_FEATURES].isna().sum()

shipping_mode                  0
days_for_shipment_scheduled    0
order_item_quantity            0
product_price                  0
customer_segment               0
market                         0
latitude                       0
longitude                      0
shipping_mode_delay_rate       0
order_hour                     0
order_day_of_week              0
order_month                    0
is_weekend                     0
order_region                   0
sales                          0
order_profit_per_order         0
order_item_profit_ratio        0
dtype: int64

In [34]:
df["shipping_mode_delay_rate"] = (
    df["shipping_mode_delay_rate"]
    .fillna(df["shipping_mode_delay_rate"].median())
)

print(df[ETA_FEATURES].isna().sum())

shipping_mode                  0
days_for_shipment_scheduled    0
order_item_quantity            0
product_price                  0
customer_segment               0
market                         0
latitude                       0
longitude                      0
shipping_mode_delay_rate       0
order_hour                     0
order_day_of_week              0
order_month                    0
is_weekend                     0
order_region                   0
sales                          0
order_profit_per_order         0
order_item_profit_ratio        0
dtype: int64


===================================================Train Test Split===============================================================

In [35]:
X = df[ETA_FEATURES]
y = df["days_for_shipping_real"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (144415, 17)
X_test: (36104, 17)
y_train: (144415,)
y_test: (36104,)


In [36]:
CATEGORICAL_FEATURES = [
    "shipping_mode",
    "customer_segment",
    "market",
    "order_region"
]

NUMERICAL_FEATURES = [
    "days_for_shipment_scheduled",
    "order_item_quantity",
    "product_price",
    "latitude",
    "longitude",
    "shipping_mode_delay_rate",
    "order_hour",
    "order_day_of_week",
    "order_month",
    "is_weekend",
    "sales",
    "order_profit_per_order",
    "order_item_profit_ratio"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES
        ),
        (
            "num",
            "passthrough",
            NUMERICAL_FEATURES
        )
    ]
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded X_train shape:", X_train_encoded.shape)
print("Encoded X_test shape:", X_test_encoded.shape)

Encoded X_train shape: (144415, 48)
Encoded X_test shape: (36104, 48)


===============================================Train XGBoost Regressor================================================================

In [40]:
# model = XGBRegressor(
#     n_estimators=300,
#     learning_rate=0.05,
#     max_depth=6,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective="reg:squarederror",
#     random_state=42,
#     n_jobs=-1
# )

# model.fit(
#     X_train_encoded,
#     y_train
# )

# print("XGBoost ETA model trained successfully.")
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_encoded,
    y_train
)

print("XGBoost ETA model trained successfully.")

XGBoost ETA model trained successfully.


In [41]:
y_pred = model.predict(X_test_encoded)

print("first 10 actual values:")
print(y_test.iloc[:10].values)

print("\n First 10 predicted values:")
print(y_pred[:10])

first 10 actual values:
[4 5 3 6 4 4 6 4 0 2]

 First 10 predicted values:
[4.0636096  4.2264156  3.6972132  4.1978555  3.8763542  4.0135937
 4.6395245  3.6609008  0.06506539 4.344909  ]


In [42]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("ETA Model Evaluation")
print("--------------------")
print(f"MAE:  {mae:.4f} days")
print(f"RMSE: {rmse:.4f} days")
print(f"R²:   {r2:.4f}")

ETA Model Evaluation
--------------------
MAE:  0.8294 days
RMSE: 1.0835 days
R²:   0.5572


In [43]:
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model,
    X_train_encoded,
    y_train,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print("5-Fold Cross-Validation R² Scores:")
print(cv_scores)

print(f"\nMean CV R²: {cv_scores.mean():.4f}")
print(f"Std CV R²:  {cv_scores.std():.4f}")

5-Fold Cross-Validation R² Scores:
[0.54115671 0.52848566 0.53741717 0.53230834 0.52548027]

Mean CV R²: 0.5330
Std CV R²:  0.0057


MAE = 0.97 days → on average, the ETA prediction is off by about 1 day.
RMSE = 1.24 days → larger prediction errors are somewhat higher than the average error.
R² = 0.424 → the model explains about 42.4% of the variation in actual shipping duration.
See this base model it is not perfect model because we dont have sufficient columns

In [ ]:
# Done run this cell 

# joblib.dump(
#     model,
#     "../models/eta_prediction/eta_model.pkl"
# )
# joblib.dump(
#     preprocessor,
#     "../models/eta_prediction/eta_preprocessor.pkl"
# )
# import os


# print(os.path.exists("../models/eta_prediction/eta_model.pkl"))
# print(os.path.exists("../models/eta_prediction/eta_preprocessor.pkl"))

['../models/eta_prediction/eta_model.pkl']

In [44]:
import os
import joblib

MODEL_DIR = "../app/ai/ml/models/eta_prediction"

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(
    model,
    os.path.join(MODEL_DIR, "eta_model.pkl")
)

joblib.dump(
    preprocessor,
    os.path.join(MODEL_DIR, "eta_preprocessor.pkl")
)

print("Updated ETA model saved successfully.")

Updated ETA model saved successfully.
